# 02 Metrics for filter pair

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import rubin_sim.maf as maf

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

## Configuration

In [ ]:
opsdb_fname = get_baseline()
run_name = os.path.split(opsdb_fname)[-1].replace(".db", "")
print(f"Using {run_name}, to be read from {opsdb_fname}")

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="02_filterpairs_maf_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

## Define slicer and metrics

In [ ]:
# %pinfo maf.FilterPairTGapsMetric

In [ ]:
# %psource maf.FilterPairTGapsMetric

In [ ]:
# Look at the filter pairs metric for random discovery
m = maf.FilterPairTGapsMetric(filter_col="band")
s = maf.HealpixSlicer(nside=64)
plotDict = {"colorMin": 0, "colorMax": 1500, "xMin": 0, "xMax": 2000}
summarystats = [maf.MedianMetric(), maf.MeanMetric(), maf.PercentileMetric(percentile=80)]
bundle = maf.MetricBundle(m, s, None, plot_dict=plotDict, summary_metrics=summarystats, run_name=run_name)

## Run the Bundle group

In [ ]:
%%time
g = maf.MetricBundleGroup({"0": bundle}, opsdb_fname, data_dir)
g.run_all()

## Access to data

In [ ]:
bundle.metric_values

In [ ]:
bundle.metric_values.compressed()

## Plot

In [ ]:
bundle.plot()

## Summary statistics

In [ ]:
bundle.summary_values

## Comparison to other

In [ ]:
# Let's compare across many runs
families = maf.archive.get_family_descriptions()
family_list = families.index.values
summary_source = "summary_2022_08_01.csv"
summaries = maf.get_metric_summaries(summary_source=summary_source)
metric_set = maf.get_metric_sets("/Users/lynnej/lsst_repos/survey_strategy/fbs_2.0/metric_sets.json")

In [ ]:
# compare filter gaps pairs metric results with total number of visits

metrics = [
    m
    for m in summaries
    if "Median FilterPairTGaps" in m
    or ("Nvisits WFD" in m and "Slicer" not in m)
    or ("Median TgapsPercent" in m)
]
mset = maf.create_metric_set_df("test", metrics)

fams = [
    "baseline",
    "rolling",
    "triplets",
    "long gaps no pairs",
    "suppress repeats",
    "good seeing",
    "bluer balance",
    "vary nes",
]
these_runs = families.explode("run").loc[fams, "run"]

fig, ax = maf.plot_run_metric(
    summaries.loc[these_runs, metrics], baseline_run="baseline_v2.0_10yrs", metric_set=mset
)
fig.set_figheight(30)
ax.set_xlim(0.7, 1.1)

In [ ]:
mset = metric_set.loc["TVS anomalies"]
fig, ax = maf.plot_run_metric(
    summaries.loc[these_runs, mset["metric"]],
    baseline_run="baseline_v2.0_10yrs",
    metric_set=mset,
    metric_label_map=mset["short_name"],
    vertical_quantity="value",
    horizontal_quantity="run",
)
fig.set_figwidth(20)